# Adaptive Quality-Aware Synthetic Data Selection Framework
## Project Walkthrough Notebook

This notebook is a presentation-friendly and executable guide to the master's thesis project. It explains the research question, the five-stage pipeline, the leakage-safe split design, ASISM, and how to run the repository safely.

> **Important:** local smoke runs use tiny synthetic fixtures and provide engineering validation only. They are not thesis results. Production experiments require CheXpert, approved artifacts, and a suitable GPU environment.

## 1. Research objective

The project studies whether synthetic chest X-rays can improve medical image classification when they are selected using multiple quality and usefulness signals instead of being added indiscriminately.

**Thesis title:** *An Adaptive Quality-Aware Synthetic Data Selection Framework for Medical Image Classification Using Diffusion Foundation Models*

The central contribution is **ASISM**: an adaptive selection module that combines similarity, image quality, uncertainty, explainability plausibility, and intended-label agreement. Its value is tested empirically rather than assumed.

## 2. End-to-end pipeline

1. **Stage 1 — Generative model:** fine-tune SDXL with LoRA on CheXpert frontal chest X-rays.
2. **Stage 2 — Synthetic generation:** generate images from auditable label recipes using the fine-tuned model.
3. **Stage 3 — ASISM:** compute independent signals, run Go/No-Go checks, tune a bounded selector, and freeze the selected set.
4. **Stage 4 — Classifier experiments:** train conditions A–E under an equal-optimizer-step fairness protocol.
5. **Stage 5 — Evaluation:** compare conditions on the untouched final patient-level holdout.

The primary endpoint is macro-AUROC over the supported disease labels. The primary comparison is **ASISM-selected synthetic data (C)** versus **matched random synthetic data (D)**.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

def find_repo_root(start=Path.cwd()):
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'configs').is_dir() and (candidate / 'scripts').is_dir():
            return candidate
    raise FileNotFoundError('Repository root not found. Open this notebook from the project checkout.')

REPO = find_repo_root()
print(f'Repository: {REPO}')
print(f'Python: {sys.version.split()[0]}')
print(f'Platform: {sys.platform}')

## 3. Repository map

- `configs/`: frozen and configurable experiment settings.
- `scripts/data/`: dataset verification, patient splits, preprocessing, and captions.
- `scripts/train/`: resumable SDXL-LoRA training.
- `scripts/generate/`: label recipes and synthetic generation.
- `scripts/asism/`: signal computation, Go/No-Go checks, tuning, freezing, and selection.
- `scripts/classify/`: auxiliary scorer and Stage 4 experimental conditions.
- `scripts/eval/`: probe evaluation, Stage 5 metrics, statistics, and comparisons.
- `docs/`: proposal, literature grounding, and frozen methodology.
- `tests/`: contracts for leakage, provenance, metrics, and the pipeline.

In [ ]:
important_paths = [
    'README.md', 'docs/proposal.md', 'docs/stages2_to_5_plan.md',
    'configs/stage1_lora_sdxl.yaml', 'configs/stage2_generation.yaml',
    'configs/stage3_asism.yaml', 'configs/stage4_classifier.yaml',
    'scripts/smoke/run_smoke_pipeline.py',
]
for rel in important_paths:
    path = REPO / rel
    print(('✓' if path.exists() else '✗'), rel)

## 4. Leakage-safe patient splits

All six partitions are created directly from one deterministic patient list. No split is derived from another.

| Split | Fraction | Purpose |
|---|---:|---|
| `gen_train` | 48% | LoRA training, reference pool, auxiliary classifier training |
| `gen_val` | 10% | LoRA monitoring, auxiliary classifier validation |
| `classifier_train` | 15% | Real component of conditions A–D and ASISM proxy training |
| `classifier_val` | 9% | Classifier early stopping, selection, and thresholds |
| `asism_tuning_heldout` | 9% | ASISM Go/No-Go and bounded proxy search |
| `final_eval_heldout` | 9% | Stage 5 final evaluation only |

The final holdout is protected by explicit access guards. Patient-level disjointness, hashes, manifests, and artifact lineage are enforced in code.

## 5. ASISM signals

- **Similarity:** hierarchical real-reference retrieval plus a separate memorization/near-duplicate indicator.
- **IQA:** generic no-reference IQA supported by domain checks such as blur, clipping, contrast, borders, anatomy, and grayscale consistency.
- **Uncertainty:** MC Dropout; moderate uncertainty may be informative, while extreme uncertainty becomes concerning when paired with other poor signals.
- **Explainability:** Grad-CAM overlap with expected regions, treated as plausibility—not proof of clinical correctness.
- **Agreement:** consistency between the conditioning intent and the auxiliary classifier's predicted disease content.

Signals are stored independently with provenance. Only signals that pass the pre-specified Go/No-Go gate may enter the selector. The selector configuration and chosen images are frozen before classifier experiments.

## 6. Experimental conditions

| Condition | Training data | Interpretation |
|---|---|---|
| A | Real only | Baseline |
| B | Real + all synthetic | Tests unfiltered augmentation |
| C | Real + ASISM-selected | Proposed approach |
| D | Real + matched-random synthetic | Controls for count and label marginals |
| E | Synthetic only | Measures synthetic-data sufficiency |

A, B, C, and E use three fixed model seeds. D uses five deterministic matched draws, each with the same three seeds: **27 total training runs**. Every run receives the same number of optimizer steps.

## 7. Environment setup

For the production workflow, use the documented RunPod environment with a persistent project root. Install dependencies from the repository root:

```bash
pip install -r environment/requirements.txt
```

CheXpert is not included. Configure Kaggle credentials, then use the repository download and verification scripts. Keep data, checkpoints, Hugging Face cache, logs, and outputs on persistent storage.

In [ ]:
# Safe read-only environment check
checks = {
    'requirements': REPO / 'environment/requirements.txt',
    'smoke overlay': REPO / 'configs/smoke_e2e.yaml',
    'dataset config': REPO / 'configs/dataset_config.yaml',
}
for name, path in checks.items():
    print(f'{name:16}: {"ready" if path.exists() else "missing"}')
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES', '(not set)'))
print('PROJECT_ROOT:', os.environ.get('PROJECT_ROOT', '(not set)'))

## 8. Local validation

Run the unit/contract tests first. Then run the local smoke phase, which builds a tiny fixture, creates six-way dev splits, preprocesses images, and generates captions without accessing thesis data.

Set the switches below to `True` only when you want to execute them.

In [ ]:
RUN_TESTS = False
RUN_LOCAL_SMOKE = False

if RUN_TESTS:
    subprocess.run([sys.executable, 'tests/run_all.py'], cwd=REPO, check=True)
else:
    print('Tests skipped. Set RUN_TESTS=True to execute.')

if RUN_LOCAL_SMOKE:
    subprocess.run([sys.executable, 'scripts/smoke/run_smoke_pipeline.py', '--phase', 'local'], cwd=REPO, check=True)
else:
    print('Local smoke skipped. Set RUN_LOCAL_SMOKE=True to execute.')

## 9. Production execution order

Follow the repository gates in order: build/freeze production splits → Stage 1 LoRA → Stage 2 pilot → visually review and approve the pilot → full generation → auxiliary classifier → five ASISM signals → Go/No-Go → bounded tuning and freeze → Stage 4 conditions → Stage 5 with an explicit final-evaluation run ID.

The maintained command reference is `docs/runpod_stage2_to_5_commands.md`. Do not bypass approval manifests or fabricate missing upstream artifacts.

In [ ]:
# Display the maintained RunPod command guide inside the notebook.
guide = REPO / 'docs/runpod_stage2_to_5_commands.md'
if guide.exists():
    text = guide.read_text(encoding='utf-8')
    print(text[:12000])
    if len(text) > 12000:
        print('\n... Open the source document for the remaining commands.')
else:
    print('RunPod guide not found.')

## 10. Inspecting outputs safely

Manifests are the audit trail: they record namespace, configuration hashes, upstream hashes, seeds, checkpoints, and completion status. Use the helper below to inspect JSON artifacts without editing them.

In [ ]:
def show_json(path, max_chars=6000):
    path = Path(path)
    if not path.is_absolute():
        path = REPO / path
    if not path.exists():
        print(f'Not found: {path}')
        return None
    payload = json.loads(path.read_text(encoding='utf-8'))
    rendered = json.dumps(payload, indent=2, ensure_ascii=False)
    print(rendered[:max_chars])
    if len(rendered) > max_chars:
        print('... output truncated')
    return payload

# Example after a dev split run:
# show_json('data/chexpert/processed/splits/dev/split_manifest_v2.json')

## 11. How to explain the contribution in a presentation

1. Synthetic medical images may help, but poor, mislabeled, redundant, or memorized images can hurt.
2. A single quality score cannot capture realism, usefulness, uncertainty, explanation plausibility, and conditioning agreement.
3. ASISM keeps these signals independent, admits them only after validation, and learns a bounded selection policy.
4. The decisive test is not C versus real-only alone; it is C versus a matched-random control D.
5. Patient-level isolation, frozen decisions, provenance hashes, equal optimization steps, and corrected statistics protect the conclusion from leakage and unfair comparisons.

## 12. Interpretation checklist

Before calling a result successful, confirm: production artifacts—not fixtures—were used; all hashes and gates passed; the final holdout was untouched before Stage 5; effective patient counts are reported after uncertain/blank-label masking; confidence intervals and effect sizes accompany corrected significance tests; and the full distribution of condition D draws is shown.